In [1]:
"""
15-tube 1 : 10 dilution series (1 µM dsDNA → 6 nM) on a STARlet

Deck
----
TIP_CAR_480_A00 @ rails 25
  [0] 1000 µL CO-RE HFT (filtered)  – tips_00   
  [1]  50 µL CO-RE filtered         – tips_01   (unused but here to be aware)

MFX_CAR_L5_base @ rails 19
  pos-0  MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)
  

MFX_CAR_L5_base @ rails 13
  pos-1  MFX_DWP_module_188042 → AGenBio_1_troughplate_100000uL_Fl (water)
  pos-0  position of PCR plate in 'Make-18ul-qPCR-plate' MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)


MFX_CAR_L5_base @ rails 7      (reserved / empty)
  pos-0  MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)
    ▸ D1 contains 1 µM dsDNA (source)

Channels
--------
2 → water load & fill           (1000 µL tip, tips_00)
3 → serial dilution transfers   (15 × 1000 µL tips, tips_00)

Author : Harley King
Date   : 2025-07-19
"""


"\n15-tube 1 : 10 dilution series (1 µM dsDNA → 6 nM) on a STARlet\n\nDeck\n----\nTIP_CAR_480_A00 @ rails 25\n  [0] 1000 µL CO-RE HFT (filtered)  – tips_00   \n  [1]  50 µL CO-RE filtered         – tips_01   (unused but here to be aware)\n\nMFX_CAR_L5_base @ rails 19\n  pos-0  MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)\n\n\nMFX_CAR_L5_base @ rails 13\n  pos-1  MFX_DWP_module_188042 → AGenBio_1_troughplate_100000uL_Fl (water)\n  pos-0  position of PCR plate in 'Make-18ul-qPCR-plate' MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)\n\n\nMFX_CAR_L5_base @ rails 7      (reserved / empty)\n  pos-0  MFX_DWP_module_188042 → Tube rack (24-well, 1.5 mL, OT-2)\n    ▸ D1 contains 1 µM dsDNA (source)\n\nChannels\n--------\n2 → water load & fill           (1000 µL tip, tips_00)\n3 → serial dilution transfers   (15 × 1000 µL tips, tips_00)\n\nAuthor : Harley King\nDate   : 2025-07-19\n"

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
# from pylabrobot.resources.hamilton.mfx_modules import MFX_DWP_module_188042
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.opentrons.tube_racks import (
    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
)
from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources import (
    TIP_50ul_w_filter, # 50 µL filtered
    HTF     # 1000 µL filtered 
)

###############################################################################
# User-adjustable parameters
###############################################################################

START_TIP = "A1"          # first 1000 µL tip to pick up (row B, col 4)
MIX_VOL   = 800           # µL, mixing volume
MIX_CYC   = 3             # cycles per tube
ASP_RATE  = None          # µL s-1, None = PLR default
DSP_RATE  = None
CHANNEL_WATER   = 1       # index = channel-2 (0-based)
CHANNEL_DILUTE  = 1       # index = channel-3 (0-based)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)

###############################################################################
# 1) carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

tiprack_1000 = HTF("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_50   = TIP_50ul_w_filter("tips_01") #  50 µL filter tips (slot-1)

# mount the racks
tip_car[0] = tiprack_1000          # OR:  tip_car[0].assign_child_resource(tiprack_1000)
tip_car[1] = tiprack_50


# --- carrier @ rail-19  ------------------------------------------------------
dwp_mod_dest   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dest")
car_19 = MFX_CAR_L5_base(
    "car_19",
    modules={
        0: dwp_mod_dest
    }
)
lh.deck.assign_child_resource(car_19, rails=19)

# labware
tuberack_dest = opentrons_24_tuberack_generic_1point5ml_snapcap_short("dest_rack")
dest_offset_x = (127.76 - tuberack_dest._size_x) / 2
dest_offset_y = (85.48  - tuberack_dest._size_y) / 2

adapter_dest = TubeRackAdapter(
    name="dest_rack_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=tuberack_dest._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=dest_offset_x,
    dy=dest_offset_y,
    dz=0,
    adapter_hole_size_x=tuberack_dest._size_x,
    adapter_hole_size_y=tuberack_dest._size_y,
    adapter_hole_size_z=tuberack_dest._size_z
)
adapter_dest.assign_child_resource(tuberack_dest)
dwp_mod_dest.assign_child_resource(adapter_dest)

# ----------carrier @ rail 13: water trough-------------------
# water reservoir
dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
car_13 = MFX_CAR_L5_base(
    "car_13",
    modules={
        1: dwp_mod_trough, # want in this position so I can add a plate to [0] position for qPCR
    }
)
lh.deck.assign_child_resource(car_13, rails=13)
trough = AGenBio_1_troughplate_100000uL_Fl("water_trough")
dwp_mod_trough.assign_child_resource(trough)

# --- carrier @ rail-7: 2mL tube with dsDNA ----------------------------
dwp_mod_src = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_src")
car_07 = MFX_CAR_L5_base(
    "car_07",
    modules={
        0: dwp_mod_src,
    }
)
lh.deck.assign_child_resource(car_07, rails=7)

tuberack_src = opentrons_24_tuberack_generic_1point5ml_snapcap_short("src_rack")
src_offset_x = (127.76 - tuberack_src._size_x) / 2
src_offset_y = (85.48  - tuberack_src._size_y) / 2

adapter_src = TubeRackAdapter(
    name="src_rack_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=tuberack_src._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=src_offset_x,
    dy=src_offset_y,
    dz=0,
    adapter_hole_size_x=tuberack_src._size_x,
    adapter_hole_size_y=tuberack_src._size_y,
    adapter_hole_size_z=tuberack_src._size_z
)
adapter_src.assign_child_resource(tuberack_src)
dwp_mod_src.assign_child_resource(adapter_src)

ImportError: cannot import name 'MFX_DWP_module_188042' from 'pylabrobot.resources.hamilton.mfx_modules' (/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot/pylabrobot/resources/hamilton/mfx_modules.py)

In [ ]:
###############################################################################
# 2) helper – tip iterator (row-major)
###############################################################################
ROW_LET = "ABCDEFGH"
def tip_sequence(start: str) -> Iterator[str]:
    sr, sc = start[0].upper(), int(start[1:])
    for r in ROW_LET[ROW_LET.index(sr):]:
        for c in range(sc if r == sr else 1, 13):
            yield f"{r}{c}"

tip_iter = tip_sequence(START_TIP)

###############################################################################
# 3) water preload (Ch-2) – 900 µL into 15 destination tubes
###############################################################################
async def preload_water():
    tip_pos = next(tip_iter)
    await lh.pick_up_tips(tiprack_1000[tip_pos], use_channels=[CHANNEL_WATER])

    # optional pre-wet
    await lh.aspirate(trough["A1"], vols=[1000], liquid_height=[2], use_channels=[CHANNEL_WATER])
    await lh.dispense(trough["A1"], vols=[1000], liquid_height=[2], use_channels=[CHANNEL_WATER])

    dest_wells = [
        "A1","A2","A3","A4","A5","A6",
        "B1","B2","B3","B4","B5","B6",
        "C1","C2","C3"
    ]
    for well in dest_wells:
        await lh.aspirate(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
        await lh.dispense(
            tuberack_dest[well], vols=[900],
            use_channels=[CHANNEL_WATER],
            liquid_height=[20], blow_out=[1]
        )
    await lh.discard_tips()  # to default waste
###############################################################################
# 4) serial dilution (Ch-3)
###############################################################################
async def serial_dilution():
    dest_wells = [
        "A1","A2","A3","A4","A5","A6",
        "B1","B2","B3","B4","B5","B6",
        "C1","C2","C3"
    ]
  

    # first transfer: dsDNA D1 → A1
    tip_pos = next(tip_iter)
    await lh.pick_up_tips(tiprack_1000[tip_pos], use_channels=[CHANNEL_DILUTE])
    try:
        await lh.aspirate(
            tuberack_src["D1"],
            vols=[100],
            use_channels=[CHANNEL_DILUTE],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            immersion_depth=[3]
        )
    except: #if fluid is low, LLDMode.Gamma throws an error
        await lh.aspirate(
            tuberack_src["D1"],
            vols=[100],
            use_channels=[CHANNEL_DILUTE],
            liquid_height=[1]
        )
        
    await lh.dispense(
        tuberack_dest["A1"],
        vols=[100],
        use_channels=[CHANNEL_DILUTE],
        mix_volume=[900],
        mix_cycles=[2],
        mix_speed=[400],
        mix_surface_following_distance=[20],
        lld_mode=[STARBackend.LLDMode.GAMMA],
        blow_out=[1]
    )
    
    await lh.discard_tips()

    # subsequent 1→2, 2→3, … transfers
    for src_well, dst_well in zip(dest_wells[:-1], dest_wells[1:]):
        tip_pos = next(tip_iter)
        await lh.pick_up_tips(tiprack_1000[tip_pos], use_channels=[CHANNEL_DILUTE])

        await lh.aspirate(
            tuberack_dest[src_well],
            vols=[100],
            liquid_height=[20],
            use_channels=[CHANNEL_DILUTE],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            immersion_depth=[3]
        )
        await lh.dispense(
            tuberack_dest[dst_well],
            vols=[100],
            use_channels=[CHANNEL_DILUTE],
            mix_volume=[900],
            mix_cycles=[2],
            mix_speed=[400],
            mix_surface_following_distance=[20],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            blow_out=[1]
        )
        await lh.discard_tips()

###############################################################################
# 5) run protocol
###############################################################################

await preload_water()
await serial_dilution()



In [ ]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.drop_tips(tiprack_1000["A1"], use_channels=[1])
# await lh.discard_tips()
await lh.stop()